# Tree + map per model

`plot_trees_and_maps_per_model(run_id, ...)` loads `results/<run_id>/` and for each model prints:

- each parseable tree (`llm_output` parsed as JSON)
- one simulation map for the chosen task (default `go_to_target_v1`)

In [2]:
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

ROOT_DIR = Path.cwd().resolve().parents[2]
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.utils import get_results_dir, get_root_dir


def load_task_env(task_id: str = "go_to_target_v1"):
    tasks_path = get_root_dir() / "src" / "tasks" / "tasks_100.json"
    tasks = json.loads(tasks_path.read_text(encoding="utf-8"))
    for t in tasks:
        if t.get("task_id") == task_id:
            world = t["world"]
            return {
                "task_id": t["task_id"],
                "task_type": t["task_type"],
                "obstacles": world.get("obstacles", []),
                "targets": world.get("targets", []),
            }
    raise ValueError(f"Task not found: {task_id}")


def parse_tree(raw: str):
    text = raw.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return json.loads(text)


def render_tree(node, indent=0):
    pad = "  " * indent
    t = node.get("type", "?")
    if t in {"sequence", "fallback"}:
        lines = [f"{pad}{t.upper()}"]
        for child in node.get("children", []):
            lines.extend(render_tree(child, indent + 1))
        return lines
    if t == "condition":
        return [f"{pad}COND {node.get('observation')} == {node.get('expected')}"]
    if t == "action":
        call = node.get("call", {})
        args = call.get("arguments", {})
        return [f"{pad}ACT  {call.get('tool_name')} {args}"]
    return [f"{pad}{t}: {node}"]


def draw_map(ax, task_env, final_spot):
    xs = [0.0]
    ys = [0.0]
    ax.plot(xs, ys, color="tab:blue", linewidth=2, marker="o", label="start")

    for o in task_env["obstacles"]:
        rect = Rectangle(
            (o["x1"], o["y1"]),
            o["x2"] - o["x1"],
            o["y2"] - o["y1"],
            facecolor="black",
            alpha=0.35,
            edgecolor="black",
        )
        ax.add_patch(rect)

    for t in task_env["targets"]:
        rect = Rectangle(
            (t["x1"], t["y1"]),
            t["x2"] - t["x1"],
            t["y2"] - t["y1"],
            facecolor="limegreen",
            alpha=0.3,
            edgecolor="green",
        )
        ax.add_patch(rect)

    ax.scatter([final_spot["x"]], [final_spot["y"]], c="red", s=70, marker="x", label="final")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Simulation map (start and final position)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")


def plot_trees_and_maps_per_model(
    run_id: str,
    task_id: str = "go_to_target_v1",
) -> None:
    """Print trees + one map figure per model folder under results/<run_id>/."""
    results_dir = get_results_dir(run_id)
    task_env = load_task_env(task_id)
    task_filename = f"task_{task_id}.json"

    if not results_dir.exists():
        raise FileNotFoundError(f"Missing results folder: {results_dir}")

    for model_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        main_path = model_dir / "main_results.json"
        task_path = model_dir / "tasks" / task_filename

        if not main_path.exists() or not task_path.exists():
            continue

        main = json.loads(main_path.read_text(encoding="utf-8"))
        task = json.loads(task_path.read_text(encoding="utf-8"))
        model_id = main.get("model_id", model_dir.name)

        print(f"===== {model_id} =====")

        parseable = 0
        for bt in task.get("behavior_trees", []):
            raw = bt.get("llm_output", "")
            try:
                tree_obj = parse_tree(raw)
            except Exception as exc:
                print(f"[tree_{bt.get('bt_index')}] not parseable: {exc}")
                continue

            parseable += 1
            print(f"[tree_{bt.get('bt_index')}]")
            root = tree_obj.get("root", {})
            print("\n".join(render_tree(root)))

        if parseable == 0:
            print("No parseable trees found.")

        fig, ax = plt.subplots(figsize=(6, 5))
        draw_map(ax, task_env, task.get("final_spot", {"x": 0.0, "y": 0.0}))
        plt.show()

ModuleNotFoundError: No module named 'pydantic'

In [ ]:
plot_trees_and_maps_per_model("bt_online_one_task_local")
